In [ ]:
# ============================================================
# LID-Net Real-Time Dehazing — Logitech C270 HD Webcam
# Complete Self-Contained Notebook
# ============================================================

!pip install torch torchvision opencv-python Pillow -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import time
import base64
import os
import sys
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display, HTML
from google.colab import files, drive
from base64 import b64decode, b64encode
from google.colab.output import eval_js
import ipywidgets as widgets

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device  : {DEVICE}")
print(f"✅ PyTorch : {torch.__version__}")
print(f"✅ OpenCV  : {cv2.__version__}")
print(f"✅ Ready!")

✅ Device  : cpu
✅ PyTorch : 2.11.0+cpu
✅ OpenCV  : 4.13.0
✅ Ready!


In [ ]:
# ============================================================
# LID-Net Full Architecture
# ============================================================

class HEBlock(nn.Module):
    """Haze Extraction Block — Paper Eq.(2)"""
    def __init__(self, in_ch, out_ch, dilation=2):
        super().__init__()
        self.static_conv  = nn.Conv2d(in_ch, out_ch, 3,
                                       padding=1, bias=False)
        self.dilated_conv = nn.Conv2d(in_ch, out_ch, 3,
                                       padding=dilation,
                                       dilation=dilation,
                                       bias=False)
        self.bn_static   = nn.BatchNorm2d(out_ch) # Renamed from bn_s
        self.bn_dilated   = nn.BatchNorm2d(out_ch) # Renamed from bn_d
        self.fuse_conv   = nn.Sequential( # Renamed from fuse
            nn.Conv2d(in_ch + out_ch, out_ch, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        X1 = self.bn_static(self.static_conv(x)) + \
             self.bn_dilated(self.dilated_conv(x)) # Updated names
        return self.fuse_conv(torch.cat([x, X1], dim=1)) # Updated name


class HRBlock(nn.Module):
    """Haze Removal Block — Paper Eq.(3)(4)(5)"""
    def __init__(self, channels):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d((None, 1))
        self.max_pool = nn.AdaptiveMaxPool2d((1, None))
        self.conv_avg = nn.Conv2d(channels, channels, 1, bias=False)
        self.conv_max = nn.Conv2d(channels, channels, 1, bias=False)
        self.conv3x3  = nn.Sequential(
            nn.Conv2d(channels, channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        X1     = self.conv_avg(self.avg_pool(x))
        X2     = self.conv_max(self.max_pool(x))
        attn   = self.sigmoid(X1) + self.sigmoid(X2)
        return self.conv3x3(x * attn)


class BCUnit(nn.Module):
    """Brightness Compensation — Paper Eq.(6)"""
    def __init__(self):
        super().__init__()

    @torch.no_grad()
    def forward(self, x):
        x   = x.clamp(0.0, 1.0)
        out = torch.zeros_like(x)
        for i in range(x.shape[0]):
            out[i] = self._process(x[i])
        return out

    def _process(self, img):
        np_img  = (img.permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
        hsv     = cv2.cvtColor(np_img, cv2.COLOR_RGB2HSV)
        Jv      = hsv[:,:,2]
        hist, _ = np.histogram(Jv.flatten(), 256, [0,256])
        cdf     = hist.cumsum()
        cdf_min = int(cdf[cdf>0].min())
        cdf_max = int(cdf.max())
        denom   = cdf_max - cdf_min
        if denom > 0:
            lut        = np.round(
                (cdf - cdf_min) / denom * 255
            ).clip(0,255).astype(np.uint8)
            hsv[:,:,2] = lut[Jv]
        rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
        return torch.from_numpy(rgb).permute(2,0,1).float().div(255).to(img.device)


class LIDNet(nn.Module):
    """Complete LID-Net — Paper Fig.1"""
    def __init__(self):
        super().__init__()
        C1,C2,C3,C4 = 8,16,32,64

        self.stem  = nn.Sequential(
            nn.Conv2d(3,C1,3,padding=1,bias=False),
            nn.BatchNorm2d(C1), nn.ReLU(inplace=True))

        # Encoder
        self.down1 = nn.Conv2d(C1,C1,2,stride=2,bias=False)
        self.he1   = HEBlock(C1,C1)
        self.down2 = nn.Conv2d(C1,C2,2,stride=2,bias=False)
        self.hr2   = HRBlock(C2)
        self.down3 = nn.Conv2d(C2,C3,2,stride=2,bias=False)
        self.he3   = HEBlock(C3,C3)
        self.down4 = nn.Conv2d(C3,C4,2,stride=2,bias=False)
        self.hr4   = HRBlock(C4)

        # Decoder
        self.up4     = nn.ConvTranspose2d(C4,C3,2,stride=2,bias=False)
        self.he_d3   = HEBlock(C3*2,C3)
        self.up3     = nn.ConvTranspose2d(C3,C2,2,stride=2,bias=False)
        self.fuse_d2 = nn.Sequential(
            nn.Conv2d(C2*2,C2,1,bias=False),
            nn.BatchNorm2d(C2),nn.ReLU(inplace=True))
        self.hr_d2   = HRBlock(C2)
        self.up2     = nn.ConvTranspose2d(C2,C1,2,stride=2,bias=False)
        self.he_d1   = HEBlock(C1*2,C1)
        self.up1     = nn.ConvTranspose2d(C1,C1,2,stride=2,bias=False)
        self.fuse_d0 = nn.Sequential(
            nn.Conv2d(C1*2,C1,1,bias=False),
            nn.BatchNorm2d(C1),nn.ReLU(inplace=True))
        self.hr_d0   = HRBlock(C1)

        self.head = nn.Sequential(
            nn.Conv2d(C1,3,3,padding=1), nn.Sigmoid())
        self.bc   = BCUnit()
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight,
                    mode='fan_out',nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, apply_bc=False):
        s0 = self.stem(x)
        e1 = self.he1(self.down1(s0))
        e2 = self.hr2(self.down2(e1))
        e3 = self.he3(self.down3(e2))
        e4 = self.hr4(self.down4(e3))
        d3 = self.he_d3(torch.cat([self.up4(e4),e3],dim=1))
        d2 = self.hr_d2(self.fuse_d2(
                torch.cat([self.up3(d3),e2],dim=1)))
        d1 = self.he_d1(torch.cat([self.up2(d2),e1],dim=1))
        d0 = self.hr_d0(self.fuse_d0(
                torch.cat([self.up1(d1),s0],dim=1)))
        out = self.head(d0)
        if apply_bc and not self.training:
            out = self.bc(out)
        return out

# Quick test
model = LIDNet()
p     = sum(x.numel() for x in model.parameters())
print(f"✅ LIDNet defined  |  {p/1e6:.4f}M parameters")
with torch.no_grad():
    t = torch.randn(1,3,64,64)
    o = model(t)
    print(f"   Input : {t.shape} → Output: {o.shape}")


✅ LIDNet defined  |  0.1829M parameters
   Input : torch.Size([1, 3, 64, 64]) → Output: torch.Size([1, 3, 64, 64])


In [ ]:
# ============================================================
# Load trained weights from Google Drive
# ============================================================

drive.mount('/content/drive')

# ── Find checkpoint ───────────────────────────────────────────────────────────
paths = [
    '/content/drive/MyDrive/lidnet/checkpoints_paper/best.pth',
    '/content/drive/MyDrive/lidnet/checkpoints_city/best.pth',
    '/content/drive/MyDrive/lidnet/checkpoints/best.pth',
]

ckpt_path = None
for p in paths:
    if Path(p).exists():
        ckpt_path = p
        break

if ckpt_path is None:
    print("⚠️  Checkpoint not found on Drive.")
    print("    Upload best.pth manually:")
    uploaded  = files.upload()
    ckpt_path = list(uploaded.keys())[0]

# ── Load ──────────────────────────────────────────────────────────────────────
model = LIDNet().to(DEVICE)
ckpt  = torch.load(ckpt_path, map_location=DEVICE)

if 'model' in ckpt:
    model.load_state_dict(ckpt['model'])
    EPOCH     = ckpt.get('epoch', 65)
    BEST_PSNR = ckpt.get('best_psnr', 13.73)
else:
    model.load_state_dict(ckpt)
    EPOCH     = 65
    BEST_PSNR = 13.73

model.eval()
print(f"✅ Model loaded  |  Epoch {EPOCH}  |  PSNR {BEST_PSNR:.4f} dB")

# ── Optimize ──────────────────────────────────────────────────────────────────
if DEVICE.type == 'cuda':
    model = model.half()
    print("✅ FP16 enabled")

try:
    d = torch.randn(1,3,256,256).to(DEVICE)
    if DEVICE.type == 'cuda': d = d.half()
    model = torch.jit.trace(model, d)
    model = torch.jit.optimize_for_inference(model)
    print("✅ TorchScript optimized")
except Exception as e:
    print(f"⚠️  TorchScript skipped: {e}")

# Warmup
with torch.no_grad():
    for _ in range(10):
        d = torch.randn(1,3,256,256).to(DEVICE)
        if DEVICE.type == 'cuda': d = d.half()
        model(d)

print("✅ Model ready!")

Mounted at /content/drive
✅ Model loaded  |  Epoch 65  |  PSNR 13.7305 dB
✅ TorchScript optimized
✅ Model ready!


In [ ]:
# ============================================================
# All helper functions for webcam + dehazing
# ============================================================

def js_to_bgr(data_url: str) -> np.ndarray:
    """Convert JS base64 data URL → BGR numpy array."""
    img_bytes = b64decode(data_url.split(',')[1])
    arr       = np.frombuffer(img_bytes, dtype=np.uint8)
    frame     = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    return frame


def bgr_to_b64(frame: np.ndarray, quality=82) -> bytes:
    """Convert BGR numpy → JPEG bytes for widget."""
    _, buf = cv2.imencode('.jpg', frame,
                          [cv2.IMWRITE_JPEG_QUALITY, quality])
    return bytes(buf)


@torch.no_grad()
def dehaze(frame_bgr: np.ndarray, size=256) -> np.ndarray:
    """
    Run LID-Net on one BGR frame.
    Returns dehazed BGR at original resolution.
    """
    oh, ow = frame_bgr.shape[:2]
    rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    rgb    = cv2.resize(rgb, (size,size),
                        interpolation=cv2.INTER_LINEAR)
    t      = torch.from_numpy(rgb).permute(2,0,1) \
                  .float().div(255.0).unsqueeze(0).to(DEVICE)
    if DEVICE.type == 'cuda':
        t  = t.half()
    out    = model(t).float().squeeze(0).clamp(0,1)
    out    = (out.permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
    out    = cv2.cvtColor(out, cv2.COLOR_RGB2BGR)
    return cv2.resize(out, (ow,oh), interpolation=cv2.INTER_LINEAR)


def add_fog(frame: np.ndarray, intensity=0.5) -> np.ndarray:
    """
    Depth-aware atmospheric fog simulation.
    Top = far = more fog, Bottom = near = less fog.
    """
    h, w   = frame.shape[:2]
    depth  = np.linspace(1.0, 0.1, h).reshape(h,1,1)
    depth  = np.tile(depth, (1,w,3)).astype(np.float32)
    beta   = intensity * 3.0
    t_map  = np.exp(-beta * depth)
    A      = np.array([210,210,205], dtype=np.float32)
    foggy  = frame.astype(np.float32)*t_map + A*(1-t_map)
    return foggy.clip(0,255).astype(np.uint8)


def build_panel(orig:    np.ndarray,
                foggy:   np.ndarray,
                dehazed: np.ndarray,
                fps:     float,
                inf_ms:  float,
                frame_n: int) -> np.ndarray:
    """
    3-panel display:
    [ ORIGINAL ] [ FOGGY INPUT ] [ LID-Net DEHAZED ]
    """
    h, w  = orig.shape[:2]
    DIV   = 3
    p1, p2, p3 = orig.copy(), foggy.copy(), dehazed.copy()

    # Label bars
    for panel, txt, col in [
        (p1, "ORIGINAL",        (160,160,160)),
        (p2, "FOGGY INPUT",     (0,130,255)),
        (p3, "LID-Net DEHAZED", (0,210,80)),
    ]:
        cv2.rectangle(panel,(0,0),(w,44),(8,8,8),-1)
        cv2.putText(panel, txt,
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX,
                    0.78, col, 2, cv2.LINE_AA)

    divider = np.full((h,DIV,3), 255, dtype=np.uint8)
    panel   = np.hstack([p1,divider,p2,divider,p3])
    pw      = panel.shape[1]

    # HUD bar
    cv2.rectangle(panel,(0,h-36),(pw,h),(0,0,0),-1)
    cv2.putText(panel,
        (f"FPS:{fps:5.1f}  |  "
         f"Inference:{inf_ms:5.1f}ms  |  "
         f"Frame:{frame_n:04d}  |  "
         f"C270 HD Webcam  |  "
         f"LID-Net [Tao et al. DSP 2024]  |  "
         f"Epoch:{EPOCH}  PSNR:{BEST_PSNR:.2f}dB"),
        (8,h-12), cv2.FONT_HERSHEY_SIMPLEX,
        0.48, (190,190,190), 1, cv2.LINE_AA)

    return panel


def build_sidebyside(foggy:   np.ndarray,
                     dehazed: np.ndarray,
                     fps:     float,
                     inf_ms:  float,
                     frame_n: int) -> np.ndarray:
    """2-panel: FOGGY | DEHAZED"""
    h, w  = foggy.shape[:2]
    left  = foggy.copy()
    right = dehazed.copy()

    cv2.rectangle(left,  (0,0),(w,44),(8,8,20),-1)
    cv2.rectangle(right, (0,0),(w,44),(5,20,5),-1)
    cv2.putText(left,  "FOGGY INPUT",
                (10,30),cv2.FONT_HERSHEY_SIMPLEX,
                0.80,(0,130,255),2,cv2.LINE_AA)
    cv2.putText(right, "LID-Net DEHAZED",
                (10,30),cv2.FONT_HERSHEY_SIMPLEX,
                0.80,(0,210,80),2,cv2.LINE_AA)

    div   = np.full((h,3,3),255,dtype=np.uint8)
    panel = np.hstack([left,div,right])
    pw    = panel.shape[1]

    cv2.rectangle(panel,(0,h-34),(pw,h),(0,0,0),-1)
    cv2.putText(panel,
        (f"FPS:{fps:.1f}  Inference:{inf_ms:.0f}ms  "
         f"Frame:{frame_n}  LID-Net [DSP 2024]"),
        (8,h-10),cv2.FONT_HERSHEY_SIMPLEX,
        0.50,(190,190,190),1,cv2.LINE_AA)
    return panel


# ── Webcam capture JS ─────────────────────────────────────────────────────────
# Specifically configured for Logitech C270 HD (720p capable)
WEBCAM_JS = """
async function c270Capture() {
  let video = document.getElementById('c270_video');

  if (!video || video.readyState < 2) {
    if (video) { video.srcObject.getTracks().forEach(t=>t.stop()); video.remove(); }

    video = document.createElement('video');
    video.id = 'c270_video';
    video.style.cssText = `
      position: fixed;
      bottom: 12px;
      right:  12px;
      width:  220px;
      height: 165px;
      z-index: 9999;
      border: 3px solid #00cc88;
      border-radius: 10px;
      box-shadow: 0 0 20px rgba(0,200,136,0.6);
    `;
    video.autoplay    = true;
    video.playsInline = true;
    document.body.appendChild(video);

    // Request HD resolution — matches C270 720p capability
    const constraints = {
      video: {
        width:     { ideal: 1280, min: 640 },
        height:    { ideal: 720,  min: 480 },
        frameRate: { ideal: 30,   max: 30  },
        facingMode: 'environment'
      }
    };

    try {
      const stream = await navigator.mediaDevices.getUserMedia(constraints);
      video.srcObject = stream;
      window.c270_stream = stream;

      // Log actual resolution
      const track    = stream.getVideoTracks()[0];
      const settings = track.getSettings();
      console.log('C270 Resolution: ' + settings.width + 'x' + settings.height);
      console.log('C270 FPS: ' + settings.frameRate);

    } catch(err) {
      // Fallback to any available camera
      console.warn('HD failed, trying default: ' + err.message);
      const stream = await navigator.mediaDevices.getUserMedia({ video: true });
      video.srcObject = stream;
      window.c270_stream = stream;
    }

    await new Promise(r => {
      video.onloadedmetadata = () => { video.play(); r(); }
    });

    // Wait for stable frame
    await new Promise(r => setTimeout(r, 800));
  }

  // Capture at full C270 resolution
  const canvas = document.createElement('canvas');
  canvas.width  = video.videoWidth  || 1280;
  canvas.height = video.videoHeight || 720;
  canvas.getContext('2d').drawImage(video, 0, 0);

  // JPEG quality 0.80 — balance quality vs transfer speed
  return canvas.toDataURL('image/jpeg', 0.80);
}
c270Capture();
"""

print("✅ All helpers ready!")
print(f"   Camera : Logitech C270 HD (720p)")
print(f"   Device : {DEVICE}")

✅ All helpers ready!
   Camera : Logitech C270 HD (720p)
   Device : cpu


In [ ]:
# ============================================================
# Real-Time C270 Webcam Dehazing Stream
# ============================================================

# ── Widgets ───────────────────────────────────────────────────────────────────
title = widgets.HTML(f"""
<div style="background:#0d0d1a;padding:14px;border-radius:10px;
            margin-bottom:10px;font-family:monospace">
  <h2 style="color:#00cc88;margin:0;font-size:20px">
    🎥 LID-Net Real-Time Road Dehazing — Logitech C270 HD
  </h2>
  <p style="color:#555;margin:4px 0 0;font-size:11px">
    Tao et al., Digital Signal Processing 154 (2024) 104673
    &nbsp;·&nbsp; Epoch {EPOCH} / 70
    &nbsp;·&nbsp; PSNR {BEST_PSNR:.2f} dB
    &nbsp;·&nbsp; 10,010 Iterations
    &nbsp;·&nbsp; Device: {str(DEVICE).upper()}
  </p>
</div>
""")

img_out  = widgets.Image(format='jpeg', width=1280, height=360)
info_out = widgets.HTML("<p style='font-family:monospace'>⏳ Starting...</p>")

fog_slider = widgets.FloatSlider(
    value=0.50, min=0.0, max=0.95, step=0.05,
    description='Fog Density:',
    style={'description_width':'125px'},
    layout=widgets.Layout(width='400px')
)
size_drop = widgets.Dropdown(
    options=[
        ('64  — Max Speed',   64),
        ('128 — Balanced',   128),
        ('192 — Quality',    192),
        ('256 — Max Quality',256),
    ],
    value=256,
    description='Model Size:',
    style={'description_width':'85px'},
    layout=widgets.Layout(width='260px')
)
view_btn = widgets.ToggleButtons(
    options=['3-Panel', 'Side-by-Side', 'Dehazed Only'],
    value='Side-by-Side',
    description='View:',
    style={'description_width':'45px',
           'button_width':'115px'}
)
stop_btn = widgets.Button(
    description='⏹ STOP',
    button_style='danger',
    layout=widgets.Layout(width='110px',height='36px')
)
snap_btn = widgets.Button(
    description='📸 Save',
    button_style='success',
    layout=widgets.Layout(width='110px',height='36px')
)

display(
    title,
    widgets.HBox([stop_btn, snap_btn,
                  fog_slider, size_drop]),
    view_btn,
    img_out,
    info_out
)

# ── State ─────────────────────────────────────────────────────────────────────
running  = {'on': True}
do_snap  = {'on': False}
snap_n   = {'n': 0}
snap_dir = Path('c270_snapshots')
snap_dir.mkdir(exist_ok=True)

def on_stop(b):
    running['on'] = False
def on_snap(b):
    do_snap['on'] = True

stop_btn.on_click(on_stop)
snap_btn.on_click(on_snap)

# ── Metrics ───────────────────────────────────────────────────────────────────
frame_n    = 0
ft_list    = []
it_list    = []

print("="*55)
print("  C270 HD WEBCAM STREAM STARTING")
print("="*55)
print("  → Allow camera access when browser prompts")
print("  → Camera preview shows bottom-right corner")
print("")
print("  Controls:")
print("  • Fog Density slider → adjust synthetic fog")
print("  • Model Size         → quality vs speed")
print("  • 3-Panel view       → Original|Foggy|Dehazed")
print("  • Side-by-Side       → Foggy|Dehazed")
print("  • Dehazed Only       → full frame output")
print("  • 📸 Save            → snapshot to Drive")
print("  • ⏹ STOP            → end session")
print("="*55)

# ── Main loop ─────────────────────────────────────────────────────────────────
with torch.no_grad():
    while running['on']:
        t0 = time.perf_counter()

        # ── Capture from C270 ─────────────────────────────────────────────────
        try:
            url   = eval_js(WEBCAM_JS)
            frame = js_to_bgr(url)
            if frame is None:
                continue
        except Exception as e:
            info_out.value = (
                f"<p style='color:orange;font-family:monospace'>"
                f"❌ Camera error: {e}<br>"
                f"→ Allow camera access and re-run this cell</p>"
            )
            break

        oh, ow = frame.shape[:2]
        fog    = fog_slider.value
        size   = size_drop.value
        view   = view_btn.value

        # ── Add synthetic fog ─────────────────────────────────────────────────
        foggy = add_fog(frame, intensity=fog)

        # ── LID-Net inference ─────────────────────────────────────────────────
        ti      = time.perf_counter()
        result  = dehaze(foggy, size=size)
        inf_ms  = (time.perf_counter()-ti)*1000
        it_list.append(inf_ms)
        if len(it_list)>30: it_list.pop(0)

        # ── FPS ───────────────────────────────────────────────────────────────
        t1 = time.perf_counter()
        ft_list.append(t1-t0)
        if len(ft_list)>30: ft_list.pop(0)
        fps     = 1.0/(sum(ft_list)/len(ft_list))
        avg_inf = sum(it_list)/len(it_list)
        frame_n += 1

        # ── Build display panel ───────────────────────────────────────────────
        if view == '3-Panel':
            panel = build_panel(
                frame, foggy, result,
                fps, avg_inf, frame_n)

        elif view == 'Side-by-Side':
            panel = build_sidebyside(
                foggy, result,
                fps, avg_inf, frame_n)

        else:
            # Dehazed Only — clean output full frame
            panel = result.copy()
            h_p, w_p = panel.shape[:2]
            cv2.rectangle(panel,(0,0),(w_p,44),(8,8,8),-1)
            cv2.putText(panel,
                "LID-Net DEHAZED OUTPUT — C270 HD",
                (10,30),cv2.FONT_HERSHEY_SIMPLEX,
                0.80,(0,210,80),2,cv2.LINE_AA)
            cv2.rectangle(panel,(0,h_p-32),(w_p,h_p),(0,0,0),-1)
            cv2.putText(panel,
                f"FPS:{fps:.1f}  Inf:{avg_inf:.0f}ms  "
                f"Frame:{frame_n}  Fog:{fog:.2f}",
                (8,h_p-10),cv2.FONT_HERSHEY_SIMPLEX,
                0.50,(190,190,190),1,cv2.LINE_AA)

        # ── Update display ────────────────────────────────────────────────────
        img_out.value  = bgr_to_b64(panel, quality=82)
        info_out.value = (
            f"<p style='font-family:monospace;font-size:13px'>"
            f"🎥 C270 HD  |  "
            f"<b>FPS: {fps:.1f}</b>  |  "
            f"Inference: <b>{avg_inf:.0f}ms</b>  |  "
            f"Fog: <b>{fog:.2f}</b>  |  "
            f"Size: <b>{size}²</b>  |  "
            f"Frame: <b>{frame_n}</b>  |  "
            f"Cam: <b>{ow}×{oh}</b>  |  "
            f"{str(DEVICE).upper()}"
            f"</p>"
        )

        # ── Save snapshot ─────────────────────────────────────────────────────
        if do_snap['on']:
            do_snap['on'] = False
            snap_n['n']  += 1
            ts   = time.strftime("%Y%m%d_%H%M%S")
            name = f"c270_snap_{snap_n['n']:03d}_{ts}"

            cv2.imwrite(str(snap_dir/f"{name}_original.jpg"), frame)
            cv2.imwrite(str(snap_dir/f"{name}_foggy.jpg"),    foggy)
            cv2.imwrite(str(snap_dir/f"{name}_dehazed.jpg"),  result)
            cv2.imwrite(str(snap_dir/f"{name}_panel.jpg"),    panel)

            # Sync to Drive
            drive_snaps = Path(
                '/content/drive/MyDrive/lidnet/c270_snapshots')
            drive_snaps.mkdir(parents=True, exist_ok=True)
            import shutil
            for f_p in snap_dir.glob(f"{name}*"):
                shutil.copy2(f_p, drive_snaps/f_p.name)

            info_out.value = (
                f"<p style='color:#00cc88;font-family:monospace'>"
                f"📸 Snapshot {snap_n['n']} saved to Drive!</p>"
            )
            print(f"📸 Snapshot {snap_n['n']} saved: {name}")

# ── Session summary ───────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  SESSION COMPLETE")
print(f"{'='*55}")
print(f"  Camera     : Logitech C270 HD")
print(f"  Frames     : {frame_n}")
print(f"  Avg FPS    : {fps:.1f}")
print(f"  Avg Inf.   : {avg_inf:.1f} ms")
print(f"  Snapshots  : {snap_n['n']}")
print(f"  Device     : {DEVICE}")
print(f"{'='*55}")

HTML(value='\n<div style="background:#0d0d1a;padding:14px;border-radius:10px;\n            margin-bottom:10px;…

ToggleButtons(description='View:', index=1, options=('3-Panel', 'Side-by-Side', 'Dehazed Only'), style=ToggleB…

Image(value=b'', format='jpeg', height='360', width='1280')

HTML(value="<p style='font-family:monospace'>⏳ Starting...</p>")

  C270 HD WEBCAM STREAM STARTING
  → Allow camera access when browser prompts
  → Camera preview shows bottom-right corner

  Controls:
  • Fog Density slider → adjust synthetic fog
  • Model Size         → quality vs speed
  • 3-Panel view       → Original|Foggy|Dehazed
  • Side-by-Side       → Foggy|Dehazed
  • Dehazed Only       → full frame output
  • 📸 Save            → snapshot to Drive
  • ⏹ STOP            → end session


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Release C270 webcam cleanly
# ============================================================

from IPython.display import display, Javascript

display(Javascript("""
if (window.c270_stream) {
    window.c270_stream.getTracks().forEach(track => {
        track.stop();
        console.log('C270 track stopped:', track.label);
    });
    const v = document.getElementById('c270_video');
    if (v) v.remove();
    window.c270_stream = null;
    console.log('✅ Logitech C270 released.');
}
"""))
print("✅ Logitech C270 HD camera released.")